## Preprocessing Tutorial

This notebook runs the preprocessing workflow of the NUTMEG project. 

Supported analysis modes:
- `RNA_ONLY`
- `MULTIOME`

Supported input sources:
- `10X_H5`
- `SEURAT_OBJECT`

Main workflow:
1. Initialize the `GRNPrep` object
2. Load the input data
3. Compute QC metrics
4. Filter cells and features
5. Run preprocessing and labeling
6. Export GRN input tables and feature matrices

Outputs are written under the selected experiment directory.


In [1]:
%load_ext rpy2.ipython

In [2]:
%%R
source("preprocessing2.R")


    an issue that caused a segfault when used with rpy2:
    https://github.com/rstudio/reticulate/pull/1188
    Make sure that you use a version of that package that includes
    the fix.
    

### Initialize workflow

Create a `GRNPrep` object to define the input data, analysis mode, output folder, and random seed used throughout the workflow.  
This object stores the Seurat object internally and manages the directories and settings used by all downstream steps.

#### Arguments

- `mode`  
  Select the analysis type. Allowed values are:
  - `"RNA_ONLY"`: use this for scRNA-seq datasets without ATAC data.
  - `"MULTIOME"`: use this for paired RNA + ATAC single-cell datasets.

- `source`  
  Select the type of input to load. Allowed values are:
  - `"10X_H5"`: load data from a 10x Genomics `.h5` file.
  - `"SEURAT_OBJECT"`: start from an existing Seurat object saved as `.rds`.

- `tenx_h5_path`  
  Path to the 10x `.h5` input file. Required only when `source = "10X_H5"`.

- `tenx_frag_path`  
  Path to the `fragments.tsv.gz` file. Required only when `mode = "MULTIOME"` and `source = "10X_H5"`.

- `rds_file`  
  Path to an existing Seurat object saved as `.rds`. Required only when `source = "SEURAT_OBJECT"`.

- `experiment`  
  Name of the output directory created by the workflow. Use a short descriptive name for the current run.

- `seed`  
  Random seed used to make results more reproducible across runs.


In [3]:
%%R
pbmc <- GRNPrep$new(
  mode = "RNA_ONLY",
  source = "SEURAT_OBJECT",
  tenx_h5_path   = "data/raw/pbmc_unsorted_3k_filtered_feature_bc_matrix.h5",
  tenx_frag_path   = "data/raw/pbmc_unsorted_3k_atac_fragments.tsv.gz",
  rds_file = "pbmc_multi_02/rds/pbmc.rds",
  experiment = "pbmc_rna_only_02",
  seed = 42
)



[GRNPrep] Initialized
 - experiment: pbmc_rna_only_02
 - mode: RNA_ONLY
 - source: SEURAT_OBJECT
 - seed: 42
 - csv_dir: /home/fotini/Desktop/lala/pbmc_rna_only_02/interm/csv
 - plot_dir: /home/fotini/Desktop/lala/pbmc_rna_only_02/interm/plots
 - feature_mtx_dir: /home/fotini/Desktop/lala/pbmc_rna_only_02/interm/feature_mtx
 - rds_dir: /home/fotini/Desktop/lala/pbmc_rna_only_02/rds



### Load data and compute QC

This step loads the input data into the workflow object and computes quality-control metrics used later for filtering.  
For RNA data, the workflow computes `log_nCount_RNA` and `percent.mt`  
In `MULTIOME` mode, ATAC QC metrics are also added, including `log_nCount_ATAC`, `log_TSS.enrichment`, and `log_nucleosome_signal`.

#### `load()` options

- `keep_standard_chromosomes`  
  Logical value (`TRUE` or `FALSE`).  
  Default: `TRUE`.  
  If `TRUE`, only standard human chromosomes are kept when loading ATAC peaks in `MULTIOME` mode, which helps remove peaks from non-standard contigs or scaffolds.  
  This option only affects multiome data with an ATAC assay.

#### `qc()` options

- `save_plots`  
  Logical value (`TRUE` or `FALSE`).  
  Default: `TRUE`.  
  If `TRUE`, the QC histograms are saved to the plots directory in addition to being displayed in the notebook.

- `filename`  
  Name of the output image file used when `save_plots = TRUE`.  
  Use this if you want to save the QC figure under a custom name.

- `force`  
  Logical value (`TRUE` or `FALSE`).  
  Default: `FALSE`.  
  If `FALSE`, the function skips recomputation when QC metrics are already present in the object.  
  If `TRUE`, QC metrics are recalculated even if they already exist.

#### Output of this step

After running these functions:
- the Seurat object is loaded into `GRNPrep`
- QC metrics are stored in the object metadata
- QC histograms are generated
- plots are optionally saved to disk

In [4]:
%%R
pbmc$load()
qc_plots <- pbmc$qc()
qc_plots


  |                                                  | 0 % ~calculating   |++                                                | 4 % ~01m 01s       |++++                                              | 8 % ~43s           |++++++                                            | 12% ~55s           |++++++++                                          | 16% ~48s           |++++++++++                                        | 20% ~49s           |++++++++++++                                      | 24% ~44s           |++++++++++++++                                    | 28% ~41s           |++++++++++++++++                                  | 32% ~39s           |++++++++++++++++++                                | 36% ~35s           |++++++++++++++++++++                              | 40% ~34s           |++++++++++++++++++++++                            | 44% ~32s           |++++++++++++++++++++++++                          | 48% ~29s           |++++++++++++++++++++++++++                        | 52% ~26s 


[load] Loading data... (source=SEURAT_OBJECT, mode=RNA_ONLY)
[load] Done. Cells=2484
[qc] QC metrics already present; skipping qc().
In addition: There were 24 warnings (use warnings() to see them)


### Filter cells and features

Filter cells using the QC metrics and remove low-detection features.  
In `RNA_ONLY` mode, the workflow filters cells and genes.  
In `MULTIOME` mode, ATAC-based cell filters and peak filtering are also applied.

#### `filter()` options

**RNA cell-level filters**

- `rna_log_ncount_min`  
  Lower bound for `log10(nCount_RNA)`.  
  Cells below this threshold are removed because they likely have very low RNA complexity.

- `rna_log_ncount_max`  
  Upper bound for `log10(nCount_RNA)`.  
  Cells above this threshold are removed as potential outliers or multiplets.

- `rna_mt_pct_max`  
  Maximum allowed mitochondrial percentage (`percent.mt`).  
  Cells above this threshold are removed because high mitochondrial signal often indicates low-quality or stressed cells.

**ATAC cell-level filters (`MULTIOME` only)**

- `atac_log_ncount_min`  
  Lower bound for `log10(nCount_ATAC)`.  
  Removes cells with very weak ATAC signal.

- `atac_log_ncount_max`  
  Upper bound for `log10(nCount_ATAC)`.  
  Removes extreme ATAC outliers.

- `atac_log_nucleo_max`  
  Maximum allowed `log10(nucleosome_signal)`.  
  High nucleosome signal can indicate poor-quality ATAC profiles.  

- `atac_log_tss_min`  
  Minimum allowed `log10(TSS.enrichment)`.  
  Cells below this threshold are removed because low TSS enrichment usually reflects poor chromatin accessibility quality.

**Feature-level filters**

- `min_cells_per_gene`  
  Keep only genes detected in at least this many cells.

- `min_cells_per_peak`  
  Keep only ATAC peaks detected in at least this many cells.  
  This option is relevant only in `MULTIOME` mode.

#### Output of this step

After filtering:
- low-quality cells are removed
- rarely detected genes are removed
- rarely detected peaks are removed in `MULTIOME` mode
- the filtered object is stored for preprocessing

In [5]:
%%R
pbmc$filter(
  rna_log_ncount_min = 3,
  rna_log_ncount_max = 4,
  rna_mt_pct_max     = 20,
  atac_log_ncount_min = 3,
  atac_log_ncount_max = 4.5,
  atac_log_nucleo_max = 0.3,
  atac_log_tss_min    = 0.0,
  min_cells_per_gene = 3,
  min_cells_per_peak = 10
)



[filter] Filtering cells & features... (mode=RNA_ONLY)
[filter] Done. Cells: 2484 -> 2484
In addition: Warning message:
Different cells and/or features from existing assay RNA 


### Preprocess and label cells

Run preprocessing, dimensionality reduction, clustering, and cell labeling.  
For RNA data, the workflow runs SCTransform and PCA.  
In `MULTIOME` mode, it also runs ATAC preprocessing, multimodal neighbor analysis, UMAP, and clustering.  
Labels can be assigned either from clusters or from an external labels file, and the final processed object is saved to disk.

#### `preprocess()` options

**Preprocessing parameters**

- `n_sct_features`  
  Number of variable genes used by SCTransform.  
  Default: `2000`.  
  Increase this if you want to include more variable genes in the RNA preprocessing step.

- `n_pca_dims`  
  Number of PCA dimensions used for downstream analysis.  
  Default: `50`.  
  This affects neighbor finding, UMAP, and clustering for RNA data.

- `n_lsi_dims`  
  Number of LSI dimensions used for ATAC data in `MULTIOME` mode.  
  Default: `20`.  
  This option is only relevant when an ATAC assay is present.

- `resolution`  
  Clustering resolution.  
  Default: `0.5`.  
  Higher values usually produce more clusters, while lower values produce fewer clusters.

- `save_plots`  
  Logical value (`TRUE` or `FALSE`).  
  Default: `TRUE`.  
  If `TRUE`, the UMAP plot is saved to the plots directory.

**Labeling parameters**

- `label_mode`  
  Defines how labels are assigned.  
  Allowed values are:
  - `"clusters"`: use Seurat cluster IDs as labels.
  - `"labels"`: import labels from an external file.  
  Default: `"clusters"`.

- `labels_column_name`  
  Name of the metadata column where labels will be stored.  
  Default: `"cell_label"`.

- `labels_file`  
  Path to a file containing cell barcodes and labels.  
  Required when `label_mode = "labels"`.

- `labels_sep`  
  Separator used in the labels file.  
  Default: `","`.

- `final_rds_filename`  
  Name of the processed Seurat object saved at the end of the step.  

#### Output of this step

After running this function:
- RNA data are normalized and reduced in dimension
- cells are clustered
- a UMAP is generated
- labels are assigned
- the processed Seurat object is saved to disk

In [6]:
%%R
pbmc$preprocess(
  n_sct_features = 2000,
  n_pca_dims     = 50,
  n_lsi_dims     = 20,
  resolution     = 0.5,
  label_mode     = "labels",
  labels_file    = "data/labels.csv",
  final_rds_filename = "pbmc.rds"
)



[preprocess] Preprocessing + labeling (mode=RNA_ONLY, labels=labels)
[preprocess] SCT/PCA already present; skipping available steps.
To use Python UMAP via reticulate, set umap.method to 'umap-learn' and metric to 'correlation'
This message will be shown once per session
[preprocess] Adding labels (mode=labels)
[preprocess] Subsetting: 2484/2484 labeled cells
[preprocess] labels.csv saved: /home/fotini/Desktop/lala/pbmc_rna_only_02/interm/csv/labels.csv
[preprocess] Labeled object saved: /home/fotini/Desktop/lala/pbmc_rna_only_02/rds/pbmc.rds
In addition: Warning message:
`aes_string()` was deprecated in ggplot2 3.0.0.
ℹ Please use tidy evaluation idioms with `aes()`.
ℹ See also `vignette("ggplot2-in-packages")` for more information.
ℹ The deprecated feature was likely used in the Seurat package.
  Please report the issue at <https://github.com/satijalab/seurat/issues>.
This warning is displayed once every 8 hours.
Call `lifecycle::last_lifecycle_warnings()` to see where this warning 

### Export GRN inputs

Generate CRE-gene and TF-CRE tables and export the feature matrices required for downstream GRN analysis.  
In `RNA_ONLY` mode, promoter-based pseudo-CREs are generated internally.

#### `grn_exports()` options

**Motif analysis parameters**

- `motif_raw_threshold`  
  Minimum raw motif score required to keep a TF-CRE edge.  
  Default: `5`.  
  Increasing this value makes motif filtering more stringent.

- `jaspar_species`  
  Species used when retrieving motifs from JASPAR.  
  Default: `"Homo sapiens"`.  
  Change this only if you are working with another organism.

- `jaspar_all_versions`  
  Logical value (`TRUE` or `FALSE`).  
  Default: `FALSE`.  
  If `TRUE`, all motif versions are retrieved from JASPAR instead of only the default set.

**CRE-gene linking parameters**

- `promoter_window_bp`  
  Promoter half-window size, in base pairs, used to define promoter regions.  
  Default: `2000`.  
  In `RNA_ONLY` mode, this controls the size of the internally generated promoter-based pseudo-CREs.

- `genes_use`  
  Optional custom vector of genes to use for GRN export.  
  Default: `NULL`.  
  If not provided, the workflow uses the SCT variable features.

- `peaks_use`  
  Optional custom vector of peaks to use for GRN export.  
  Default: `NULL`.  
  If not provided in `MULTIOME` mode, the workflow uses ATAC variable features.

- `link_peak_assay`  
  Name of the assay used for peaks during peak-gene linking.  
  Default: `"ATAC"`.  
  This option is relevant only in `MULTIOME` mode.

- `link_expression_assay`  
  Name of the assay used for gene expression during peak-gene linking.  
  Default: `"SCT"`.  
  This option is relevant only in `MULTIOME` mode.

**Output file parameters**

- `peaks_genes_filename`  
  Output filename for the CRE-gene table.  
  Default: `"peaks_genes.csv"`.

- `tf_peak_scores_filename`  
  Output filename for the TF-CRE table.  
  Default: `"tf_peak_scores.csv"`.

#### Output of this step

After running this function:
- a CRE-gene table is exported
- a TF-CRE score table is exported
- raw and processed feature matrices are written to disk
- feature name files and barcode files are generated for downstream GRN analysis

In [7]:
%%R
pbmc$grn_exports(
  motif_raw_threshold = 5,
  jaspar_species = "Homo sapiens"
)



[grn_exports] Building base GRN exports... (mode=RNA_ONLY)
[grn_exports] Gene-side set = 2000 genes.
[grn_exports] RNA_ONLY: using promoter-only pseudo-CREs.
[grn_exports] RNA_ONLY CRE->Gene edges: 14986 | unique CREs=14986 | unique genes=1770
[grn_exports] Motif matching on CREs (JASPAR2020) ... this can take a while.
[grn_exports] TF->CRE edges after filtering: 351937 (raw_score > 5)

[grn_exports] Export complete:
 - CRE->Gene: /home/fotini/Desktop/lala/pbmc_rna_only_02/interm/csv/peaks_genes.csv
 - TF->CRE: /home/fotini/Desktop/lala/pbmc_rna_only_02/interm/csv/tf_peak_scores.csv

[grn_exports] Exporting feature matrices...
Features: TFs=59 Genes=1770 (RNA-only, no CRE features)
 [RNA_ONLY] Skipping CRE features.
[grn_exports] Feature matrices complete.
